## Setting GPU

In [2]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


TensorFlow version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Importing Dataset from Drive

In [3]:
from google.colab import drive

import os
drive.mount('/content/drive')
!unzip -q "/content/drive/MyDrive/datasets.zip" -d "/content/"
print("Unzip complete!")

Mounted at /content/drive
Unzip complete!


## Setting Paths

In [4]:

BASE_DIR = "/content/datasets/face_dataset"
train_dir = os.path.join(BASE_DIR, "train")
val_dir = os.path.join(BASE_DIR, "val")
test_dir = os.path.join(BASE_DIR, "test")

for d in [train_dir, val_dir, test_dir]:
    if os.path.exists(d):
        print(f"Found: {d}")
        print(f"Contains: {os.listdir(d)}")
    else:
        print(f"NOT Found: {d}")

Found: /content/datasets/face_dataset/train
Contains: ['fake', 'real']
Found: /content/datasets/face_dataset/val
Contains: ['fake', 'real']
Found: /content/datasets/face_dataset/test
Contains: ['fake', 'real']


## Imports

In [7]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

## Setting Training parameters

In [8]:
IMG_SIZE=(224,224)
EPOCHS=20
BATCH_SIZE=32

In [9]:
from tensorflow.keras.applications.efficientnet import preprocess_input

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

val_gen = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

print("Class indices:", train_gen.class_indices)

Found 9023 images belonging to 2 classes.
Found 1933 images belonging to 2 classes.
Class indices: {'fake': 0, 'real': 1}


## Efficient Net Face Model

In [10]:
base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

# Freeze backbone
base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,213,668 (16.07 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

## Model save checkpoint

In [11]:
checkpoint = ModelCheckpoint(
    "/content/face_model_stage1.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

## Model training

In [12]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=[checkpoint, early_stop]
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
282/282 ━━━━━━━━━━━━━━━━━━━━ 0s 438ms/step - accuracy: 0.6487 - loss: 0.6281
Epoch 1: val_accuracy improved from -inf to 0.80807, saving model to /content/face_model_stage1.keras
282/282 ━━━━━━━━━━━━━━━━━━━━ 175s 515ms/step - accuracy: 0.6489 - loss: 0.6279 - val_accuracy: 0.8081 - val_loss: 0.4481
Epoch 2/20
282/282 ━━━━━━━━━━━━━━━━━━━━ 0s 366ms/step - accuracy: 0.7869 - loss: 0.4656
Epoch 2: val_accuracy improved from 0.80807 to 0.83652, saving model to /content/face_model_stage1.keras
282/282 ━━━━━━━━━━━━━━━━━━━━ 108s 382ms/step - accuracy: 0.7869 - loss: 0.4656 - val_accuracy: 0.8365 - val_loss: 0.3960
Epoch 3/20
282/282 ━━━━━━━━━━━━━━━━━━━━ 0s 360ms/step - accuracy: 0.8011 - loss: 0.4260
Epoch 3: val_accuracy improved from 0.83652 to 0.84997, saving model to /content/face_model_stage1.keras
282/282 ━━━━━━━━━━━━━━━━━━━━ 105s 372ms/step - accuracy: 0.8011 - loss: 0.4260 - val_accuracy: 0.8500 - val_loss: 0.3687
Epoch 4/20
282/282 ━━━━━━━━━━━━━━━━━━━━ 0s 363ms/step - accur

### Fine tuning Model

In [13]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_fine = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=5
)

Epoch 1/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 162s 472ms/step - accuracy: 0.7438 - loss: 0.5026 - val_accuracy: 0.8484 - val_loss: 0.3255
Epoch 2/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 110s 390ms/step - accuracy: 0.8394 - loss: 0.3581 - val_accuracy: 0.8557 - val_loss: 0.3069
Epoch 3/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 109s 387ms/step - accuracy: 0.8472 - loss: 0.3336 - val_accuracy: 0.8670 - val_loss: 0.2887
Epoch 4/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 107s 378ms/step - accuracy: 0.8513 - loss: 0.3256 - val_accuracy: 0.8748 - val_loss: 0.2759
Epoch 5/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 109s 387ms/step - accuracy: 0.8683 - loss: 0.3016 - val_accuracy: 0.8815 - val_loss: 0.2635


## Save Model

In [15]:
model.save("/content/drive/MyDrive/face_model_final_tuned.keras")

In [16]:
!pip freeze

absl-py==1.4.0
accelerate==1.12.0
access==1.1.10.post3
affine==2.4.0
aiofiles==24.1.0
aiohappyeyeballs==2.6.1
aiohttp==3.13.3
aiosignal==1.4.0
aiosqlite==0.22.1
alabaster==1.0.0
albucore==0.0.24
albumentations==2.0.8
ale-py==0.11.2
alembic==1.18.4
altair==5.5.0
annotated-doc==0.0.4
annotated-types==0.7.0
antlr4-python3-runtime==4.9.3
anyio==4.12.1
anywidget==0.9.21
apsw==3.51.2.0
apswutils==0.1.2
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
array_record==0.8.3
arrow==1.4.0
arviz==0.22.0
astropy==7.2.0
astropy-iers-data==0.2026.2.16.0.48.25
astunparse==1.6.3
atpublic==5.1
attrs==25.4.0
audioread==3.1.0
Authlib==1.6.8
autograd==1.8.0
babel==2.18.0
backcall==0.2.0
beartype==0.22.9
beautifulsoup4==4.13.5
betterproto==2.0.0b6
bigframes==2.35.0
bigquery-magics==0.10.3
bleach==6.3.0
blinker==1.9.0
blis==1.3.3
blobfile==3.2.0
blosc2==4.0.0
bokeh==3.7.3
Bottleneck==1.4.2
bqplot==0.12.45
branca==0.8.2
brotli==1.2.0
CacheControl==0.14.4
cachetools==7.0.1
catalogue==2.0.10
certifi==2026.1.4
cf

In [17]:
!pip freeze | grep -E "tensorflow|keras|numpy|scikit|matplotlib|Pillow|seaborn"


keras==3.10.0
keras-hub==0.21.1
keras-nlp==0.21.1
matplotlib==3.10.0
matplotlib-inline==0.2.1
matplotlib-venn==1.1.2
numpy==2.0.2
scikit-image==0.25.2
scikit-learn==1.6.1
seaborn==0.13.2
tensorflow==2.19.0
tensorflow-datasets==4.9.9
tensorflow-hub==0.16.1
tensorflow-metadata==1.17.3
tensorflow-probability==0.25.0
tensorflow-text==2.19.0
tensorflow_decision_forests==1.12.0
tf_keras==2.19.0
